# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muneeb-khokhar/flyrank-ml-track/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item on one report_date (finest grain in fact_content_daily_performance) — aggregated up to one row per content item over a rolling 30-day window for feature-building.
Table(s): fact_content_daily_performance (mid-panel partition month=2026-03), joined to dim_content and dim_clients.

Time window: 30 days ending at the partition's max report_date, with a trailing prior 30-day window for trend comparison — not the sealed final month.

In [4]:
%pip -q install duckdb huggingface_hub

import os
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

# Proof 1: grain check
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c
    FROM {REL} GROUP BY 1,2,3 HAVING c > 1 LIMIT 5
""").df()
print("Grain violations (should be empty):", len(grain_check))

# Proof 2: row count + date span
span = con.sql(f"SELECT COUNT(*) AS rows, MIN(report_date) AS start, MAX(report_date) AS end FROM {REL}").df()
print(span)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain violations (should be empty): 0
      rows      start        end
0  9841378 2026-03-01 2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Label/proxy: is_declining (impressions dropped ≥20%, prev30 vs last30). Context: client_hash_id, content_hash_id. Excluded: gsc_avg_position on zero-impression days (0 = no data, not rank zero). Five features (all knowable at decision time, computed on prev30 only): imp_prev30, clk_prev30, pos_prev30, days_active_prev30, content_age_days.

In [5]:
REL_CONTENT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"

features = con.sql(f"""
    WITH bounds AS (SELECT MAX(report_date) AS end_d FROM {REL}),
    windowed AS (
        SELECT client_hash_id, content_hash_id,
               SUM(CASE WHEN report_date >  b.end_d - INTERVAL 30 DAY THEN gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN report_date <= b.end_d - INTERVAL 30 DAY THEN gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN report_date <= b.end_d - INTERVAL 30 DAY THEN gsc_clicks      ELSE 0 END) AS clk_prev30,
               AVG(CASE WHEN report_date <= b.end_d - INTERVAL 30 DAY AND gsc_avg_position > 0
                        THEN gsc_avg_position END)                                                     AS pos_prev30,
               SUM(CASE WHEN report_date <= b.end_d - INTERVAL 30 DAY AND gsc_impressions > 0
                        THEN 1 ELSE 0 END)                                                             AS days_active_prev30
        FROM {REL} f, bounds b
        GROUP BY 1,2
        HAVING imp_prev30 >= 10
    )
    SELECT * FROM windowed
""").df()

content_meta = con.sql(f"""
    WITH bounds AS (SELECT MAX(report_date) AS end_d FROM {REL})
    SELECT content_hash_id,
           DATE_DIFF('day', content_created_date, (SELECT end_d FROM bounds)) AS content_age_days
    FROM {REL_CONTENT}
""").df()

features = features.merge(content_meta, on="content_hash_id", how="left")
print(f"{len(features):,} content items with enough prior-window history")
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

60,333 content items with enough prior-window history


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_prev30,pos_prev30,days_active_prev30,content_age_days
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1120.0,20.0,0.0,3.350000,1.0,396
1,client_73cda7b4e4f265ea,content_05434271b257bb68,1366.0,55.0,0.0,3.272727,1.0,396
2,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2693.0,77.0,0.0,5.636364,1.0,396
3,client_73cda7b4e4f265ea,content_712c365258cee05c,5825.0,223.0,0.0,3.892377,1.0,396
4,client_73cda7b4e4f265ea,content_8935ed68eca88b01,4675.0,113.0,0.0,5.654867,1.0,396


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Proof 3: availability, filtered with IS NOT NULL / IS TRUE per this card's own warning that the flag can be NULL, not just FALSE.

In [6]:
REL_CLIENTS = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')"

before = len(features)
avail = con.sql(f"""
    SELECT f.*
    FROM features f
    JOIN {REL_CLIENTS} c ON f.client_hash_id = c.client_hash_id
    WHERE c.ga4_data_start IS NOT NULL
""").df()
print(f"Before availability filter: {before:,} | After: {len(avail):,}")

# --- THE TRAP ---
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

features["is_declining"] = (features["imp_last30"] < 0.8 * features["imp_prev30"]).astype(int)
honest_cols = ["imp_prev30", "clk_prev30", "pos_prev30", "days_active_prev30", "content_age_days"]
model_df = features.dropna(subset=honest_cols)

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_idx, te_idx = next(gss.split(model_df, groups=model_df["client_hash_id"]))

def quick_auc(cols):
    X = model_df[cols].fillna(0)
    y = model_df["is_declining"]
    m = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    m.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    return roc_auc_score(y.iloc[te_idx], m.predict_proba(X.iloc[te_idx])[:, 1])

honest_auc = quick_auc(honest_cols)
print(f"Honest AUC (prev-30 features only): {honest_auc:.3f}")

leaky_cols = honest_cols + ["imp_last30"]
leaky_auc = quick_auc(leaky_cols)
print(f"Leaky AUC (with imp_last30 added): {leaky_auc:.3f}  <- jumps toward 1.0, the model is scoring its own answer")

print(f"\nKept, honest result: {honest_auc:.3f} AUC using only prev-30 features.")

Before availability filter: 60,333 | After: 46,016
Honest AUC (prev-30 features only): 0.486
Leaky AUC (with imp_last30 added): 1.000  <- jumps toward 1.0, the model is scoring its own answer

Kept, honest result: 0.486 AUC using only prev-30 features.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice is a single mid-panel month from an unbalanced panel — client history depth varies (some clients have 17 months, some 3), and a one-month slice slightly favors clients already active during March 2026. It can't tell me anything about seasonality.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.